# DDM Model Compilation and Testing Workflow

This notebook provides step-by-step instructions for compiling and testing the DDM (Drift Diffusion Model) JAGS module.


## Prerequisites

Ensure you have the required dependencies installed:
- JAGS 4.3.0+
- ONNX Runtime 1.23.2+ (Linux x64)
- py2jags 0.1.0+
- Python 3.11+ with scikit-learn


## Step 1: Generate Module Code


In [28]:
!cd /home/jovyan/project && ./scripts/generate-module models/ddm.jnnx


Found files:
  Metadata: models/ddm.jnnx/metadata.json
  ONNX: models/ddm.jnnx/model.onnx
  Scalers: models/ddm.jnnx/scalers.pkl

/opt/conda/envs/pymc/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Metadata loaded: ddm_emulator
Scalers loaded: 4 parameters

✓ ONNX Runtime found in tmp/onnxruntime-linux-x64-1.23.2

Output directory: tmp/ddm.jnnx_build

Generating module code...
Copied ONNX model to: tmp/ddm.jnnx_build/model.onnx
Generated: tmp/ddm.jnnx_build/ddm_emulator.cc

Generating Makefile...
Generated: tmp/ddm.jnnx_build/Makefile

Module generation complete!

To compile and install:
  cd tmp/ddm.jnnx_build
  make
  sudo make install


## Step 2: Compile the Module


In [29]:
!cd tmp/ddm.jnnx_build && make


g++ -fPIC -O2 -Wall -std=c++17 -I/usr/include/JAGS -I../../tmp/onnxruntime-linux-x64-1.23.2/include -c ddm_emulator.cc -o ddm_emulator.o
g++ -shared -o ddm_emulator.so ddm_emulator.o -ljags -L../../tmp/onnxruntime-linux-x64-1.23.2/lib -lonnxruntime


In [30]:
!cd tmp/ddm.jnnx_build && sudo make install


mkdir -p /usr/lib/x86_64-linux-gnu/JAGS/modules-4/
cp ddm_emulator.so /usr/lib/x86_64-linux-gnu/JAGS/modules-4//


## Step 3: Test the Module


In [31]:
import py2jags
import onnxruntime as ort
import numpy as np

# Test DDM function
model_string = '''
model {
    result <- ddm(0.5, 0.5, 0.5)
    dummy ~ dnorm(0, 1)
}
'''

result = py2jags.run_jags(
    model_string=model_string,
    data_dict={'n': 1},
    nchains=1, nsamples=1, nadapt=0, nburnin=0,
    monitorparams=['result'],
    modules=['ddm_emulator'],
    verbosity=0
)

print('DDM output:', [result.get_samples(f'result_{i+1}')[0] for i in range(3)])


DDM output: [0.496954, 0.320284, 0.0849938]


## Step 4: Validate Against Python ONNX


In [32]:
import onnxruntime as ort
import numpy as np

# Load ONNX model
session = ort.InferenceSession('models/ddm.jnnx/model.onnx')
test_input = np.array([[0.5, 0.5, 0.5]], dtype=np.float32)
result = session.run(['output'], {'input': test_input})
print('Python ONNX output:', result[0][0].tolist())


Python ONNX output: [0.496954083442688, 0.32028400897979736, 0.08499377965927124]


## Step 5: Compare Results


In [33]:
# Run both tests and compare
import py2jags
import onnxruntime as ort
import numpy as np

# JAGS evaluation
model_string = '''
model {
    result <- ddm(0.5, 0.5, 0.5)
    dummy ~ dnorm(0, 1)
}
'''

jags_result = py2jags.run_jags(
    model_string=model_string,
    data_dict={'n': 1},
    nchains=1, nsamples=1, nadapt=0, nburnin=0,
    monitorparams=['result'],
    modules=['ddm_emulator'],
    verbosity=0
)

jags_output = [jags_result.get_samples(f'result_{i+1}')[0] for i in range(3)]

# Python ONNX evaluation
session = ort.InferenceSession('models/ddm.jnnx/model.onnx')
test_input = np.array([[0.5, 0.5, 0.5]], dtype=np.float32)
onnx_result = session.run(['output'], {'input': test_input})
onnx_output = onnx_result[0][0].tolist()

# Compare results
print('JAGS output:  ', jags_output)
print('Python output:', onnx_output)
print('Differences:  ', [abs(jags_output[i] - onnx_output[i]) for i in range(3)])

# Check if results match within machine precision
max_diff = max([abs(jags_output[i] - onnx_output[i]) for i in range(3)])
if max_diff < 1e-6:
    print('SUCCESS: Results match within machine precision')
else:
    print('ERROR: Results differ significantly')


JAGS output:   [0.496954, 0.320284, 0.0849938]
Python output: [0.496954083442688, 0.32028400897979736, 0.08499377965927124]
Differences:   [8.344268798143872e-08, 8.97979735015042e-09, 2.0340728754120185e-08]
SUCCESS: Results match within machine precision


## Expected Results

Both outputs should match within machine precision (differences < 1e-6).